# 06-4 실습 · 여러 시스템을 엮는 에이전트

교안 `06-4_실습_멀티도구에이전트.html`과 같은 순서입니다. 탭 1 ~ 4는 여행 상담(예약 · 운항 · 규정 · 환불), 아래 M1 ~ M5는 감기 진료(환자 조회 · 검색 · 재진 예약)로 복습합니다.

## 0. 개요 · RAG 에이전트와 진짜 에이전트

> **전제** 06-3에서 RAG를 도구로 쥔 에이전트를 만들었습니다. 그런데 질문 대부분이 **검색 → 계산 → 발송** 같은 순서로 흘러, 고정 그래프로도 거의 똑같이 만들 수 있었습니다.

### 에이전트가 필요한 순간

에이전트의 가치는 **성격이 다른 시스템 여러 개를 질문마다 다르게 엮을 때** 드러납니다. 이번 실습의 상담 에이전트는 문서 검색만 하지 않고 예약 시스템 · 날짜 계산 · 운항 시스템 · 환불 계산 · 예약 취소까지 다룹니다.

| 시스템 | 도구 | 06-3에 있었나 |
|---|---|---|
| 예약 DB | `get_booking(customer)` 고객 예약 조회 | 없음 |
| 달력 | `days_until(date_str)` 오늘부터 며칠 남았나 | 없음 (고객이 "3일 전"이라고 직접 말해야 했음) |
| 규정 문서 (RAG) | `search_policy(query)` travel_docs.csv 검색 | 있음 (search_docs) |
| 환불 계산 | `calc_refund(price, days_before, rule)` 상품(T15) · 숙소(T16) 규정 따로 | 상품 규정만 |
| 운항 시스템 | `check_ferry(date_str)` 솔바람페리 운항 여부 | 없음 |
| 예약 처리 | `cancel_booking(booking_id)` 실제 취소 (승인 필요) | 없음 (발송만) |

### 같은 에이전트, 질문마다 다른 길

| 고객 질문 | 에이전트가 고를 경로 (예) |
|---|---|
| "내 예약 지금 취소하면 얼마 돌려받아?" | get_booking → days_until → search_policy → calc_refund(상품) + calc_refund(숙소) |
| "내일 페리 결항이라던데 내 여행은?" | get_booking → check_ferry → search_policy(결항 규정) |
| "별빛전망대는 언제 쉬어요?" | search_policy 하나 |
| "출발까지 며칠 남았어?" | get_booking → days_until |
| (앞 대화에 이어) "그럼 취소해줘" | 기억에서 예약번호 → cancel_booking (승인 대기) |

> **이번 실습에서 볼 것** 코드에는 어떤 경로도 적혀 있지 않습니다. 도구 6개, 프롬프트 규칙, 체크포인터만 있습니다. 경로는 매 질문마다 LLM이 만듭니다. 탭 5에서 "이걸 고정 그래프로 짜면?"을 비교합니다.

> **예약 · 운항 시스템은 가짜** 실제 DB · API 대신 파이썬 dict로 흉내 냅니다. 오늘 날짜도 2026-11-11로 고정합니다. 실습마다 같은 결과가 나오게 하기 위해서입니다. 실무에서는 이 함수 안만 실제 API 호출로 바꾸면 됩니다.

## 1. 가짜 시스템과 도구 6개

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community faiss-cpu pandas

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."   # 본인 키

### 규정 문서 (RAG) 준비

In [ ]:
import pandas as pd
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import init_chat_model

df = pd.read_csv("travel_docs.csv")
docs = [Document(page_content=f"{r['title']}\n{r['body']}", metadata={"doc_id": r["doc_id"]})
        for _, r in df.iterrows()]
db = FAISS.from_documents(docs, OpenAIEmbeddings())
llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

### 가짜 예약 · 운항 시스템

In [ ]:
from datetime import date

TODAY = date(2026, 11, 11)          # 실습용으로 오늘 날짜 고정

BOOKINGS = {                        # 예약 DB 흉내
    "kim":  {"id": "B-1001", "product": "솔바람섬 별빛 2박3일", "depart": "2026-11-14",
             "tour": 250000, "hotel": 140000, "hotel_name": "파도소리 리조트"},
    "lee":  {"id": "B-1002", "product": "솔바람섬 별빛 2박3일", "depart": "2026-11-12",
             "tour": 250000, "hotel": 140000, "hotel_name": "파도소리 리조트"},
    "choi": {"id": "B-1003", "product": "노을항 미식 투어", "depart": "2026-11-25",
             "tour": 170000, "hotel": 80000, "hotel_name": "노을스테이 호텔"},
}
FERRY = {"2026-11-12": "결항 (기상 악화)"}   # 운항 시스템 흉내: 없는 날은 정상 운항
CANCELLED = []                              # 취소 처리 기록

| 고객 | 출발 | 오늘 기준 | 포인트 |
|---|---|---|---|
| kim | 2026-11-14 | 3일 전 | 상품(T15 30%)과 숙소(T16 50%) 환불률이 다름 |
| lee | 2026-11-12 | 내일 | 그날 페리 결항 |
| choi | 2026-11-25 | 14일 전 | 노을항 상품이라 페리와 무관 |

### 도구 6개

In [ ]:
from langchain.tools import tool


@tool(parse_docstring=True)
def get_booking(customer: str) -> str:
    """고객의 예약 내역(예약번호, 상품, 출발일, 상품 · 숙소 금액)을 조회한다. 고객 본인 예약에 대한 질문이면 먼저 사용한다.

    Args:
        customer: 고객 이름 (예: kim)
    """
    b = BOOKINGS.get(customer)
    if not b:
        return f"{customer} 고객의 예약이 없습니다."
    return (f"{b['id']} | {b['product']} | 출발 {b['depart']} | "
            f"상품 {b['tour']:,}원 + 숙소({b['hotel_name']}) {b['hotel']:,}원")


@tool(parse_docstring=True)
def days_until(date_str: str) -> str:
    """오늘부터 주어진 날짜까지 며칠 남았는지 계산한다. 출발 며칠 전인지 알아야 할 때 사용한다.

    Args:
        date_str: 날짜 (YYYY-MM-DD)
    """
    n = (date.fromisoformat(date_str) - TODAY).days
    return f"{date_str}까지 {n}일 남음 (오늘 {TODAY})"


@tool(parse_docstring=True)
def search_policy(query: str) -> str:
    """바람길여행 · 숙소 · 페리 규정과 여행지 정보를 검색한다. 규정이나 장소 정보가 필요하면 사용한다.

    Args:
        query: 검색할 내용 (예: 페리 결항 환불)
    """
    found = db.similarity_search(query, k=3)
    return "\n\n".join(f"[{d.metadata['doc_id']}] {d.page_content}" for d in found)


@tool(parse_docstring=True)
def calc_refund(price: int, days_before: int, rule: str) -> str:
    """환불액을 규정대로 계산한다. 상품 비용과 숙소 비용은 규정이 달라 따로 계산한다.

    Args:
        price: 금액 (원)
        days_before: 출발(체크인) 며칠 전인지
        rule: "tour"(상품, T15) 또는 "hotel"(숙소, T16)
    """
    if rule == "tour":
        rate = 1.0 if days_before >= 10 else 0.8 if days_before >= 5 else 0.3 if days_before >= 2 else 0.0
        name = "상품(T15)"
    else:
        rate = 1.0 if days_before >= 7 else 0.5 if days_before >= 3 else 0.0
        name = "숙소(T16)"
    return f"{name} {days_before}일 전 → {rate:.0%} 환불 = {int(price * rate):,}원"


@tool(parse_docstring=True)
def check_ferry(date_str: str) -> str:
    """해당 날짜의 솔바람페리(노을항 ↔ 솔바람섬) 운항 여부를 확인한다. 페리 · 결항 · 날씨로 인한 운항 질문이면 사용한다.

    Args:
        date_str: 날짜 (YYYY-MM-DD)
    """
    return f"{date_str} 솔바람페리: {FERRY.get(date_str, '정상 운항 (하루 4회)')}"


@tool(parse_docstring=True)
def cancel_booking(booking_id: str) -> str:
    """예약을 실제로 취소한다. 되돌릴 수 없으므로 고객이 취소해 달라고 분명히 말할 때만 사용한다.

    Args:
        booking_id: 예약번호 (예: B-1001)
    """
    CANCELLED.append(booking_id)
    return f"{booking_id} 취소 완료"


TOOLS = [get_booking, days_until, search_policy, calc_refund, check_ferry, cancel_booking]

### 도구만 먼저 테스트

In [ ]:
print(get_booking.invoke({"customer": "kim"}))
print(days_until.invoke({"date_str": "2026-11-14"}))
print(calc_refund.invoke({"price": 250000, "days_before": 3, "rule": "tour"}))
print(calc_refund.invoke({"price": 140000, "days_before": 3, "rule": "hotel"}))
print(check_ferry.invoke({"date_str": "2026-11-12"}))
print(check_ferry.invoke({"date_str": "2026-11-14"}))

> **kim의 정답은 145,000원** 상품 75,000원 + 숙소 70,000원. 같은 "3일 전"이어도 상품은 30%, 숙소는 50%입니다. LLM이 문서를 읽고 한 번에 계산하면 놓치기 쉬운 부분이라 rule을 나눠 두 번 계산하게 했습니다.

| 도구 설계 포인트 | 이유 |
|---|---|
| get_booking 결과에 예약번호 · 출발일 · 금액을 한 줄로 | 다음 도구(days_until, calc_refund, cancel_booking)의 인자를 LLM이 그대로 옮겨 쓸 수 있게 |
| days_until을 따로 | LLM은 날짜 뺄셈도 틀립니다. 06-1의 "계산은 도구로" |
| cancel_booking docstring에 "분명히 말할 때만" | 환불액만 물었는데 취소해 버리는 사고를 설명서 단계에서 막음 (최종 안전망은 승인) |

## 2. 에이전트 만들기 · 규칙과 안전장치

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (HumanInTheLoopMiddleware, ModelCallLimitMiddleware,
                                         ToolCallLimitMiddleware)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

PROMPT = """너는 바람길여행 상담 에이전트다. 오늘은 2026-11-11이다.
1. 고객 본인 예약에 대한 질문이면 get_booking으로 예약부터 확인한다.
2. 출발 며칠 전인지는 days_until로 계산한다. 직접 계산하지 않는다.
3. 규정은 search_policy로 찾고 근거 문서 번호를 [T15]처럼 적는다. 바람결여행(T22)은 다른 회사다.
4. 환불액은 calc_refund로 상품(tour)과 숙소(hotel)를 따로 계산해 합친다.
5. 페리 운항 질문은 check_ferry로 확인한다.
6. 고객이 취소해 달라고 분명히 말할 때만 cancel_booking을 쓴다.
7. 도구 결과와 문서에 없는 내용은 '확인 후 안내드리겠습니다'라고 한다."""

agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=PROMPT,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "reject"]}}),
        ToolCallLimitMiddleware(tool_name="cancel_booking", run_limit=1, exit_behavior="end"),
        ModelCallLimitMiddleware(run_limit=10),
    ],
)

| 설정 | 막는 것 |
|---|---|
| 규칙 1 · 5 · 6 | 질문에 맞는 시스템을 고르게 하는 **판단 기준**. 경로를 정하는 게 아니라 "이럴 땐 이걸"만 알려 줌 |
| 규칙 2 · 4 | 날짜 · 금액 계산을 LLM이 직접 하지 못하게 |
| 규칙 7 | 모르는 걸 지어내지 않게 (결항 시 상품 환불 규정은 문서에 없음 → 탭 3) |
| HumanInTheLoop (cancel_booking) | 실제 취소는 상담원 승인 뒤에만 |
| ToolCallLimit (cancel_booking 1회) | 한 번 실행에 두 건 이상 취소하는 사고 |
| ModelCallLimit (10회) | 도구를 6개나 주면 헤매며 반복할 수 있음 → 비용 상한 |
| checkpointer | "그럼 취소해줘"처럼 앞 대화를 이어받기 + 승인 대기 저장 |

### 실행 과정 보기 도우미

In [ ]:
# 이번 질문에서 부른 도구(→)와 결과(←), 최종 답 또는 멈춤을 보여 준다
def ask(thread, question):
    cfg = {"configurable": {"thread_id": thread}}
    before = len(agent.get_state(cfg).values.get("messages", [])) if agent.get_state(cfg).values else 0
    r = agent.invoke({"messages": [("user", question)]}, cfg)
    print(f"Q. {question}")
    for m in r["messages"][before + 1:]:
        for tc in getattr(m, "tool_calls", []) or []:
            print("  →", tc["name"], tc["args"])
        if m.type == "tool":
            print("     ←", m.content.splitlines()[0][:70])
    if "__interrupt__" in r:
        req = r["__interrupt__"][0].value["action_requests"][0]
        print("  [멈춤] 승인 대기:", req["name"], req["args"])
    else:
        print("  A.", r["messages"][-1].content)
    print()
    return cfg, r

`before`로 이번 질문 전까지의 메시지 수를 세어 두고, 그 뒤에 생긴 메시지만 보여 줍니다. 같은 thread에서 이어 물을 때 앞 대화의 도구 호출이 다시 찍히지 않게 하기 위해서입니다.

## 3. 질문 4개, 경로 4개

같은 에이전트에 성격이 다른 질문 네 개를 보냅니다. 출력의 **→ 줄(도구 호출)이 질문마다 어떻게 달라지는지**를 봅니다.

### ① 환불액: 예약 → 날짜 → 규정 → 계산 두 번

In [ ]:
cfg_kim, r = ask("상담-kim", "고객 kim: 내 예약 지금 취소하면 얼마 돌려받아?")

1. **고객은 날짜도 금액도 말하지 않았다.** 06-3에서는 "39만 원, 3일 전"을 고객이 말해야 했습니다. 이제 에이전트가 예약 시스템과 달력에서 직접 알아냅니다.
2. **앞 도구의 결과가 다음 도구의 인자가 됐다.** get_booking의 출발일 → days_until, 금액 → calc_refund. 이 연결을 LLM이 했습니다.
3. **calc_refund를 한 번에 두 개.** 상품과 숙소를 동시에 요청했습니다(병렬 도구 호출, 06-1 탭 2).

### ② 페리 결항: 예약 → 운항 → 규정

In [ ]:
cfg_lee, r = ask("상담-lee", "고객 lee: 내일 페리 결항이라던데 내 여행은 어떻게 돼?")

calc_refund를 부르지 않았습니다. 이 질문에는 계산이 필요 없기 때문입니다. 그리고 문서에 없는 "결항 시 상품 처리"는 규칙 7대로 지어내지 않고 확인하겠다고 했습니다.

### ③ 장소 정보: 검색 하나

In [ ]:
ask("상담-guest", "별빛전망대는 언제 쉬어요?")

### ④ 남은 날짜: 예약 → 날짜

In [ ]:
ask("상담-choi", "고객 choi: 내 여행 출발까지 며칠 남았어?")

### 네 경로 비교

| 질문 | get_booking | days_until | search_policy | calc_refund | check_ferry |
|---|---|---|---|---|---|
| ① 환불액 | ✓ | ✓ | ✓ | ✓✓ |  |
| ② 페리 결항 | ✓ |  | ✓ |  | ✓ |
| ③ 장소 정보 |  |  | ✓ |  |  |
| ④ 남은 날짜 | ✓ | ✓ |  |  |  |

> **이것이 에이전트다움** 네 질문이 모두 다른 도구 조합 · 다른 순서였습니다. 06-3은 표의 search_policy 열만 거의 항상 채워졌습니다. RAG는 여섯 도구 중 하나일 뿐입니다.

## 4. 대화를 이어받아 실제로 취소하기

①의 kim이 같은 thread에서 이어 말합니다. 예약번호를 다시 말하지 않습니다.

In [ ]:
cfg_kim, r = ask("상담-kim", "그럼 취소해줘")

1. **get_booking을 다시 부르지 않았다.** 체크포인터가 앞 대화(예약 조회 결과)를 함께 넘겨서, LLM이 B-1001을 기억에서 꺼냈습니다.
2. **"그럼"을 알아들었다.** 앞에서 145,000원 환불 안내를 들은 뒤의 취소라는 맥락을 이해한 것입니다.
3. **취소 직전에 멈췄다.** 되돌릴 수 없는 일이라 상담원 승인을 기다립니다. CANCELLED는 아직 비어 있습니다.

In [ ]:
print("승인 전 취소 기록:", CANCELLED)
r = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), cfg_kim)
print(r["messages"][-1].content)
print("승인 후 취소 기록:", CANCELLED)

### 거부: 결항 고객에게는 다른 길을 먼저

lee는 페리 결항 때문에 취소하려 합니다. 하지만 결항이면 승선권 무료 변경이 가능하니, 상담원이 취소를 막고 대안을 안내하게 합니다.

In [ ]:
cfg_lee, r = ask("상담-lee", "그냥 취소해줘")
if "__interrupt__" in r:
    r = agent.invoke(Command(resume={"decisions": [{
        "type": "reject",
        "message": "결항이면 날짜 변경이 무료입니다. 취소 전에 날짜 변경 가능성부터 안내하세요.",
    }]}), cfg_lee)
print(r["messages"][-1].content)
print("취소 기록:", CANCELLED)

> **거부 사유가 다음 행동을 바꾼다** 06-3에서는 거부 사유를 읽고 "다시 검색"했다면, 여기서는 **취소 대신 대안 안내**로 행동 자체를 바꿨습니다. 상담원은 지시만 하고, 어떤 도구로 어떻게 할지는 에이전트가 정합니다.

## 5. 이걸 고정 그래프로 짜면?

탭 3 · 4의 다섯 상황을 05처럼 고정 그래프로 만든다고 해 봅시다. 질문 유형마다 분기를 미리 그려야 합니다.

`classify → 환불? / 페리? / 장소? / 날짜? / 취소? / 환불+취소? / 페리+환불? / …`

|  | 고정 그래프 | 에이전트 |
|---|---|---|
| 새 질문 유형 ("숙소만 취소하면?") | 분기 · 노드 추가, 엣지 수정 | 도구 그대로, 필요하면 프롬프트 한 줄 |
| 질문 하나에 여러 의도 ("결항이면 환불은 얼마?") | 조합마다 경로를 따로 그려야 함 | 필요한 도구를 이어 부름 |
| 새 시스템 연결 ("렌터카 조회") | 그 시스템을 쓰는 모든 경로 수정 | 도구 하나 추가 |
| 결과 예측 · 테스트 | **쉬움** (경로가 코드에 있음) | 어려움 (매번 다를 수 있음) |
| 비용 | **적음** | 도구가 많을수록 LLM 호출 증가 |

> **그래서 06-3의 라우터와 함께 쓴다** 단순하고 자주 오는 질문(장소 정보)은 고정 RAG로, 여러 시스템을 엮어야 하는 질문은 이 에이전트로. 06-3 탭 4의 `agent` 노드 자리에 이번 에이전트를 넣으면 됩니다.

### 06-3과 06-4 정리

|  | 06-3 RAG 에이전트 | 06-4 멀티 도구 에이전트 |
|---|---|---|
| 도구 성격 | 검색 중심 + 계산 · 발송 | 예약 · 달력 · 문서 · 계산 · 운항 · 처리, 서로 다른 시스템 |
| 고객이 줘야 하는 정보 | 가격, 며칠 전 | 이름만 (나머지는 시스템에서) |
| 경로 | 거의 항상 검색 → 계산 → 발송 | 질문마다 다름 |
| 대화 이어받기 | 필수 아님 | "그럼 취소해줘"처럼 핵심 |
| 거부 후 행동 | 다시 검색 | **다른 행동으로 전환** (취소 → 대안 안내) |

> **에이전트의 가치는 "검색을 잘하는 것"이 아니라 "여러 시스템을 질문에 맞게 엮는 것"이다.** 경로를 그릴 수 있으면 고정 그래프, 경로가 질문마다 달라지면 에이전트.

### M1. 환자 조회 · 체온 판정 도구

의원의 진료 보조 에이전트를 만듭니다. 이번에는 문서 검색 외에 **환자 기록 시스템**(환자ID로 조회)과 **재진 예약 시스템**을 다룹니다. 먼저 조회 · 판정 도구 두 개를 만듭니다.

| 도구 | 시스템 | 하는 일 |
|---|---|---|
| `get_patient(patient_id)` | 환자 기록 | 환자ID로 나이 · 증상 · 체온 · 진단 · 처방 조회 |
| `check_fever(temp)` | 계산 | 38.5도 이상 고열, 37.5도 이상 미열 |
| `search_records(query)` | RAG | 비슷한 과거 진료 검색 (M2) |
| `book_followup(patient_id, days_later)` | 예약 | 재진 예약 (승인 필요, M2) |

In [ ]:
from datetime import timedelta

df2 = pd.read_csv("감기환자 진료 데이터.csv")
records = [Document(page_content=r["진료내용"], metadata={"환자ID": r["환자ID"]}) for _, r in df2.iterrows()]
db2 = FAISS.from_documents(records, OpenAIEmbeddings())
BOOKED = []   # 재진 예약 기록

# TODO: get_patient(patient_id) - df2에서 한 줄 찾아 "P004 | 38세 남 | 증상 ... | 체온 39.1도 | 7일째 | 진단 ... | 처방 ..." 형태로
# TODO: check_fever(temp)
# TODO: get_patient.invoke({"patient_id": "P004"}), 없는 환자 "P999"도 테스트

### M2. 검색 · 재진 예약 도구와 에이전트

비슷한 과거 진료 검색(RAG)과 재진 예약 도구를 더해, 도구 4개를 쥔 에이전트를 만듭니다. 재진 예약은 의사 승인 뒤에만, 한 번 실행에 1건만 허용합니다.

In [ ]:
# TODO: search_records(query), book_followup(patient_id, days_later) - 예약일 = TODAY + days_later
# TODO: PROMPT2 규칙 (환자 질문이면 get_patient 먼저, 체온은 check_fever, 비슷한 사례는 search_records, 재진은 요청 시에만)
# TODO: clinic = create_agent(... HITL(book_followup), ToolCallLimit(book_followup 1, "end"), ModelCallLimit(10), checkpointer)

### M3. 질문 3개, 경로 3개

의사가 성격이 다른 질문 세 개를 합니다. 각각 어떤 도구를 어떤 순서로 부를지 **먼저 예측**하고 실행해 비교합니다.

| 질문 | 내 예측 (실행 전에 적기) |
|---|---|
| "P004 환자 상태 요약해줘" |  |
| "P027이랑 비슷한 환자들은 무슨 처방 받았어?" |  |
| "급성 인두염은 보통 어떤 처방이야?" |  |

In [ ]:
# TODO: ask("의사-1", ...), ask("의사-2", ...), ask("의사-3", ...)

### M4. 이어서 "재진 잡아줘" · 승인과 거부

M3의 의사-1이 같은 thread에서 "그 환자 3일 뒤로 재진 잡아줘"라고 이어 말합니다. 환자ID를 다시 말하지 않습니다. 승인해 예약하고, 다른 thread에서는 거부해 행동이 바뀌는지 봅니다.

In [ ]:
# TODO: ask("의사-1", "그 환자 3일 뒤로 재진 잡아줘") → 멈춤 확인 → approve → BOOKED 확인
# TODO: ask("의사-2", "그 환자 내일 재진 잡아줘") → reject(message="미열이고 3일째라 재진 대신 경과 관찰 안내로") → 최종 답 확인

### M5. 설계 판단: 고정 그래프 vs 에이전트

원장님이 "에이전트는 예측이 안 돼서 불안하다. 고정 그래프로 바꾸자"고 합니다. M3 · M4의 질문을 근거로 판단을 정리합니다.

| 질문 유형 | 필요한 도구 조합 | 고정 그래프로 만들 수 있나 | 추천 |
|---|---|---|---|
| 특정 환자 상태 요약 |  |  |  |
| 비슷한 환자 처방 |  |  |  |
| 진단명별 일반 처방 |  |  |  |
| 재진 예약 |  |  |  |
| "P004랑 비슷한 환자 중 고열인 사람 재진 잡아줘" |  |  |  |

| 질문 유형 | 필요한 도구 조합 | 고정 그래프로 | 추천 |
|---|---|---|---|
| 특정 환자 상태 요약 | get_patient → check_fever | 쉬움 | 고정 (자주 오고 경로 일정) |
| 비슷한 환자 처방 | get_patient → search_records | 쉬움 | 고정 |
| 진단명별 일반 처방 | search_records | 쉬움 | 고정 RAG |
| 재진 예약 | (get_patient) → book_followup + 승인 | 가능 | 고정 + interrupt |
| 복합 요청 | get_patient → search_records → 각 환자 get_patient · check_fever → book_followup 여러 번 | **경로가 결과에 따라 달라져 어려움** | 에이전트 (+ 예약 횟수 제한 조정) |

---
## 예시 답안 (막힐 때만 보기)

#### M1 예시 답안

In [ ]:
@tool(parse_docstring=True)
def get_patient(patient_id: str) -> str:
    """환자ID로 최근 진료 기록(나이, 증상, 체온, 진단, 처방)을 조회한다. 특정 환자에 대한 질문이면 먼저 사용한다.

    Args:
        patient_id: 환자ID (예: P004)
    """
    row = df2[df2["환자ID"] == patient_id]
    if row.empty:
        return f"{patient_id} 환자 기록이 없습니다."
    r = row.iloc[0]
    return (f"{patient_id} | {r['나이']}세 {r['성별']} | 증상 {r['증상']} | 체온 {r['체온']}도 | "
            f"{r['증상지속일']}일째 | 진단 {r['진단명']} | 처방 {r['처방']}")


@tool(parse_docstring=True)
def check_fever(temp: float) -> str:
    """체온으로 열 상태를 판정한다. 체온이 있으면 반드시 사용한다.

    Args:
        temp: 체온 (섭씨, 예: 38.2)
    """
    if temp >= 38.5:
        return f"{temp}도: 고열 (38.5도 이상)"
    if temp >= 37.5:
        return f"{temp}도: 미열 (37.5~38.4도)"
    return f"{temp}도: 정상 체온"


print(get_patient.invoke({"patient_id": "P004"}))
print(get_patient.invoke({"patient_id": "P999"}))
print(check_fever.invoke({"temp": 39.1}))

#### M2 예시 답안

In [ ]:
@tool(parse_docstring=True)
def search_records(query: str) -> str:
    """증상이 비슷한 과거 진료 기록을 검색한다. 비슷한 사례의 진단 · 처방을 참고할 때 사용한다.

    Args:
        query: 나이 · 증상 · 체온 등 (예: 6세 재채기 가래 미열)
    """
    found = db2.similarity_search(query, k=3)
    return "\n".join(f"[{d.metadata['환자ID']}] {d.page_content}" for d in found)


@tool(parse_docstring=True)
def book_followup(patient_id: str, days_later: int) -> str:
    """재진을 예약한다. 의사가 재진을 잡아 달라고 할 때만 사용한다.

    Args:
        patient_id: 환자ID (예: P004)
        days_later: 오늘부터 며칠 뒤 (예: 3)
    """
    when = TODAY + timedelta(days=days_later)
    BOOKED.append((patient_id, str(when)))
    return f"{patient_id} 재진 예약 완료: {when}"


PROMPT2 = """너는 감기 진료 보조 에이전트다. 오늘은 2026-11-11이다.
1. 특정 환자 질문이면 get_patient로 기록부터 확인한다.
2. 체온 판정은 check_fever로 한다.
3. 비슷한 사례가 필요하면 search_records로 찾고 근거 환자ID를 [P004]처럼 적는다.
4. 재진 예약은 의사가 잡아 달라고 할 때만 book_followup을 쓴다.
5. 최종 판단은 의사가 한다는 점을 잊지 않는다."""

clinic = create_agent(
    model=llm, tools=[get_patient, check_fever, search_records, book_followup], system_prompt=PROMPT2,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={"book_followup": {"allowed_decisions": ["approve", "reject"]}}),
        ToolCallLimitMiddleware(tool_name="book_followup", run_limit=1, exit_behavior="end"),
        ModelCallLimitMiddleware(run_limit=10),
    ],
)
agent = clinic        # 탭 2의 ask() 도우미를 그대로 쓰기 위해

#### M3 예시 답안

In [ ]:
ask("의사-1", "P004 환자 상태 요약해줘")
ask("의사-2", "P027이랑 비슷한 환자들은 무슨 처방 받았어?")
ask("의사-3", "급성 인두염은 보통 어떤 처방이야?")

#### M4 예시 답안

In [ ]:
cfg1, r = ask("의사-1", "그 환자 3일 뒤로 재진 잡아줘")
if "__interrupt__" in r:
    r = clinic.invoke(Command(resume={"decisions": [{"type": "approve"}]}), cfg1)
print("예약 기록:", BOOKED)

cfg2, r = ask("의사-2", "그 환자 내일 재진 잡아줘")
if "__interrupt__" in r:
    r = clinic.invoke(Command(resume={"decisions": [{
        "type": "reject", "message": "미열이고 3일째라 재진 대신 경과 관찰 안내로 바꿔 주세요"}]}), cfg2)
print(r["messages"][-1].content)
print("예약 기록:", BOOKED)

#### M5 예시 답안